# Blume-Capel model with muca in the chrystal field

In [6]:
using Pkg
Pkg.activate(dirname(@__DIR__))
Pkg.instantiate()

using MonteCarloX
using SpinSystems
using Random
using Plots

  Activating project at `~/.julia/dev/MonteCarloX.jl/examples`


Build the Blume-Capel model with Hamiltonian
$$
H = -J\sum_{ij}s_i s_j +\Delta\sum s_i^2
$$

In [ ]:
L=8
sys = BlumeCapel([8,8])
# weights for the sum_pair_interacionts (can use delta_pair_interaction)
logweight_pair = BoltzmannLogWeight(β=1.0)
# weights for the sum_spin2
logweight_spin2 = MulticanonicalLogWeight(0:1:length(sys.spins))
# so define a final logweight as the sum of the two
@inline logweight(H::NTuple) = logweight_pair(H[1]) + logweight_spin2(H[2])

BlumeCapel(Int8[1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  1, 1, 1, 1, 1, 1, 1, 1, 1, 1], Graphs.SimpleGraphs.SimpleGraph{Int64}(128, [[2, 8, 9, 57], [1, 3, 10, 58], [2, 4, 11, 59], [3, 5, 12, 60], [4, 6, 13, 61], [5, 7, 14, 62], [6, 8, 15, 63], [1, 7, 16, 64], [1, 10, 16, 17], [2, 9, 11, 18]  …  [47, 54, 56, 63], [48, 49, 55, 64], [1, 49, 58, 64], [2, 50, 57, 59], [3, 51, 58, 60], [4, 52, 59, 61], [5, 53, 60, 62], [6, 54, 61, 63], [7, 55, 62, 64], [8, 56, 57, 63]]), [[2, 8, 9, 57], [1, 3, 10, 58], [2, 4, 11, 59], [3, 5, 12, 60], [4, 6, 13, 61], [5, 7, 14, 62], [6, 8, 15, 63], [1, 7, 16, 64], [1, 10, 16, 17], [2, 9, 11, 18]  …  [47, 54, 56, 63], [48, 49, 55, 64], [1, 49, 58, 64], [2, 50, 57, 59], [3, 51, 58, 60], [4, 52, 59, 61], [5, 53, 60, 62], [6, 54, 61, 63], [7, 55, 62, 64], [8, 56, 57, 63]], Int8[-1, 0, 1], 1, 0, 128, 64, 64)

In [ ]:
rng = Xoshiro(42)
alg = Metropolis(rng, logweight)

# thermalization
for i in 1:1000
    spin_flip!(sys, alg)
end
# reset alg -> which also resets the histogram
reset!(alg) 
# recording
for i in 1:10_000_000
    spin_flip!(sys, alg)
end
# update weights
update_weights!(logweight_spin2)
# continue
# for pmuca: simply do everything on MulticanonicalLogWeight ...
# this then becomes very similar for parallel tempering as an extra layer that just helps exchange states


